# Seq Len Hidden State Debug


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd

from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.experiments.model_benchmarks.seq_len_debug_utils import (
    plot_recurrent_effective_rank_trajectory,
    plot_recurrent_metric_per_layer,
    plot_recurrent_metric_per_head,
    run_recurrent_state_trajectory_tracking,
    run_hidden_state_tracking,
    summarize_hidden_state_by_seqlen,
)
from pfns.utils import get_default_device

plt.rcParams['figure.dpi'] = 400

## Config


In [ ]:
EXPERIMENT = {
    'name': 'seq_len_hidden_state_debug',
    'num_classes': 5,
    'num_features': 10,
    'num_test_samples': 100,
    'num_repetitions': 50,
    'data_generation_seed': 42,
    'seqlen_list': [250, 500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000],
}

MODEL_NAMES = [
    'Linear_Attention_Non_Causal',
    'Linear_Attention_Causal_Comb_ST'
]

TRAINING_CONTEXT_LENGTH = 1_000

device = str(get_default_device())
models_to_compare = get_models_from_names(MODEL_NAMES)

print(f'Device: {device}')
print(f'Models: {list(models_to_compare)}')
print(f'Repetitions: {EXPERIMENT["num_repetitions"]}')
print(f'Seq lens: {EXPERIMENT["seqlen_list"]}')

## Run Hidden-State Tracking


In [ ]:
hidden_state_df = run_hidden_state_tracking(
    experiment=EXPERIMENT,
    models_to_compare=models_to_compare,
    device=device,
)

print('hidden_state_df rows:', len(hidden_state_df))
hidden_state_df.head()

## Summaries and Plots


In [ ]:
summary_df = summarize_hidden_state_by_seqlen(hidden_state_df)

print('summary_df rows:', len(summary_df))
summary_df.head()

### Metric Guide
- `abs_max_mean`: average largest absolute value in each recurrent-state head matrix.
- `frobenius_norm_mean`: per-head Frobenius norm of `recurrent_state` matrices.
- `spectral_norm_mean`: per-head spectral norm (largest singular value) of `recurrent_state` matrices.
- `effective_rank_mean`: DeltaProduct-style effective rank `exp(H(sigma / ||sigma||_1))` of each recurrent-state head matrix.


In [ ]:
all_tensors = sorted(summary_df['tensor_name'].astype(str).unique().tolist())
print(f'Recurrent-state tensors available: {len(all_tensors)}')
for name in all_tensors:
    print(' ', name)


In [ ]:
plot_specs = [
    (
        'abs_max_mean',
        plot_recurrent_metric_per_head,
        {'title_prefix': 'Recurrent-state abs max vs sequence length'},
    ),
    (
        'spectral_norm_mean',
        plot_recurrent_metric_per_head,
        {'title_prefix': 'Recurrent-state spectral norm vs sequence length'},
    ),
    (
        'effective_rank_mean',
        plot_recurrent_metric_per_layer,
        {
            'title_prefix': 'Recurrent-state effective rank vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
        },
    ),
]

for metric, plot_fn, extra_kwargs in plot_specs:
    if metric not in summary_df.columns:
        continue
    plot_fn(summary_df, metric=metric, **extra_kwargs)


## Effective-Rank Trajectory
Tracks recurrent-state effective rank over token positions within one long sequence, rather than averaging a final state over multiple `seqlen` values.

This section currently supports FLA backbones only.

In [ ]:
TRAJECTORY_SEQLEN = 16_384
TRAJECTORY_REP = 0
TRAJECTORY_LAYERS = [11]
TRAJECTORY_HEADS = [0, 1, 2, 3]
TRAJECTORY_BOUNDARIES = None

trajectory_df = run_recurrent_state_trajectory_tracking(
    experiment=EXPERIMENT,
    models_to_compare=models_to_compare,
    device=device,
    seqlen=TRAJECTORY_SEQLEN,
    rep=TRAJECTORY_REP,
)

print('trajectory_df rows:', len(trajectory_df))
trajectory_df.head()

plot_recurrent_effective_rank_trajectory(
    trajectory_df,
    layer_indices=TRAJECTORY_LAYERS,
    head_indices=TRAJECTORY_HEADS,
    training_context_length=TRAINING_CONTEXT_LENGTH,
    boundary_positions=TRAJECTORY_BOUNDARIES,
)
